# Email Assistant

We are building an Email assistant which reads our email and then perform action either reply to the email or draft a new email.

For this, we would need to authenticate the user and then based on authentication assistant would perform further action.

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

EmailContext to authenticate the user

In [2]:
from dataclasses import dataclass

@dataclass
class EmailContext:
    email_address: str = "saket@example.com"
    password: str = "password123"

In [3]:
from langchain.agents import AgentState

class AuthenticatedState(AgentState):
    authenticated: bool

Lets create some tools for agent to work on

In [5]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def check_mail() -> str:
    """Check the inbox for recent emails"""
    return """
    Hello Saket,
    I'm going to be in town next week and was wondering if we could grab a coffee?
    - best, Jane (jane@example.com)
    """

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send a response email"""
    return f"Email sent to {to} with subject {subject} and body {body}"

@tool
def authenticate(email: str, password: str, runtime: ToolRuntime) -> Command:
    """Authenticate the user with the given email and password"""
    if email == runtime.context.email_address and password == runtime.context.password:
        return Command(update={
            "authenticated": True,
            "messages": [ToolMessage(
                "Successfully authenticated",
                tool_call_id=runtime.tool_call_id)]
        })
    else:
        return Command(update={
            "authenticated": False,
            "messages": [ToolMessage(
                "Authentication failed",
                tool_call_id=runtime.tool_call_id
            )]
        })

In [6]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def dynamic_tool_call(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Allow read inbox and send email tools only if user provided correct email and password"""
    authenticated = request.state.get("authenticated")

    if authenticated:
        tools = [check_mail, send_email]
    else:
        tools = [authenticate]

    request = request.override(tools=tools)
    return handler(request)

In [8]:
from langchain.agents.middleware import dynamic_prompt

authenticated_prompt = """You are a helpful assistant that can check the inbox and send emails.
Your first step after authentication is to check the inbox."""

unauthenticated_prompt = "You are a helpful assistant that can authenticate users."

@dynamic_prompt
def dynamic_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on authentication status"""
    authenticated = request.state.get("authenticated")

    if authenticated:
        return authenticated_prompt
    else:
        return unauthenticated_prompt

In [9]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

agent = create_agent(
    "claude-haiku-4-5",
    tools=[authenticate, check_mail, send_email],
    checkpointer=InMemorySaver(),
    state_schema=AuthenticatedState,
    context_schema=EmailContext,
    middleware=[
        dynamic_tool_call,
        dynamic_prompt,
        HumanInTheLoopMiddleware(
            interrupt_on={
                "authenticate": False,
                "check_mail": False,
                "send_email": True
            }
        )
    ]
)

In [10]:
from langchain.messages import HumanMessage
config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="saket@example.com, password123")]},
    context=EmailContext(),
    config=config
)

print(response['messages'][-1].content)

/Users/saketmunda/Work/Learn/langchain-atoz/lca-lc-foundations/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=EmailContext(email_addres... password='password123'), input_type=EmailContext])
  return self.__pydantic_serializer__.to_python(


Perfect! You have one email in your inbox:

**From:** Jane (jane@example.com)

**Message:**
"Hello Saket, I'm going to be in town next week and was wondering if we could grab a coffee? - best, Jane"

Would you like me to help you reply to this email or perform any other action?


In [12]:
response = agent.invoke(
    {"messages": [HumanMessage(content="any draft is fine. don't check back.")]},
    context=EmailContext(),
    config=config
)

print(response['messages'][-1].content)

[{'id': 'toolu_01D7HZeJryxECLRepic97WUj', 'caller': {'type': 'direct'}, 'input': {'to': 'jane@example.com', 'subject': 'Re: Coffee next week', 'body': "Hi Jane,\n\nThanks for reaching out! I'd love to grab a coffee with you next week. Let me know what day and time works best for you, and we can find a place that's convenient.\n\nLooking forward to catching up!\n\nBest,\nSaket"}, 'name': 'send_email', 'type': 'tool_use'}]


In [13]:
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Hi Jane,

Thanks for reaching out! I'd love to grab a coffee with you next week. Let me know what day and time works best for you, and we can find a place that's convenient.

Looking forward to catching up!

Best,
Saket


In [14]:
from langgraph.types import Command

response = agent.invoke(
    Command(
        resume={"decisions": [
            {
                "type": "approve"
            }
        ]}
    ),
    config=config
)

print(response["messages"][-1].content)

Done! I've sent a reply to Jane confirming that you'd like to grab coffee and asking her about her availability.


In [15]:
from pprint import pprint

pprint(response)

{'authenticated': True,
 'messages': [HumanMessage(content='saket@example.com, password123', additional_kwargs={}, response_metadata={}, id='83defd92-54c9-44ef-8006-025a204edfa8'),
              AIMessage(content=[{'text': "I'll authenticate you with the provided email and password.", 'type': 'text'}, {'id': 'toolu_01Wf1Eb87NwBA8iJVpKkGXfC', 'caller': {'type': 'direct'}, 'input': {'email': 'saket@example.com', 'password': 'password123'}, 'name': 'authenticate', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_01FhgFDE9mmnEhziRPjoDbqj', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 592, 'output_tokens': 86, 'server_tool_use': None, 'service_tier': 'standard'}, 'stop_details': None, 'model_name': 'c